# 🚦 VN-Traffic-Density — Stage 3: Finetune YOLOv8s trên Data Việt Nam

**Pipeline:** COCO Pretrained → BDD100K Finetune ✅ → **VN Data Finetune**

**Mục tiêu giai đoạn này:**
- Adapt model từ giao thông quốc tế (BDD100K) sang giao thông đô thị hỗn hợp Việt Nam
- Gradual unfreeze backbone theo epoch (0–14: freeze 10 → 15–34: freeze 7 → 35+: freeze 3)
- 4 classes (giữ nguyên thứ tự từ Stage 2): `car (0)`, `truck (1)`, `bus (2)`, `motor (3)`

---
**Kaggle inputs cần attach:**
```
1. data-vn        → /kaggle/input/data-vn/dataset_aug/
                       ├── images/  ← ảnh .jpg (chưa split)
                       └── labels/  ← .txt YOLO format

2. stage2-model   → /kaggle/input/stage2-model/
                       └── best.pt  ← checkpoint Stage 2 BDD100K
```

## 0. Kiểm tra GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU             : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 1. Cài đặt thư viện

In [ ]:
!pip install ultralytics==8.3.* --quiet
!pip install supervision --quiet

## 2. Khai báo đường dẫn

In [ ]:
import os, random, shutil
import numpy as np
from pathlib import Path

# ─── CẤU HÌNH ĐƯỜNG DẪN ───────────────────────────────────────────────────────
# Dataset VN: slug của dataset khi Add Data trên Kaggle
VN_DATA_SLUG   = 'data-vn'       # → /kaggle/input/data-vn/dataset_aug/
MODEL_SLUG     = 'stage2-model'  # → /kaggle/input/stage2-model/best.pt
STAGE2_PT_NAME = 'best.pt'       # tên file checkpoint Stage 2

# Tỉ lệ chia train/val/test
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# TEST_RATIO  = 0.15 (phần còn lại)

SEED = 42
# ──────────────────────────────────────────────────────────────────────────────

# Seed cố định
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Đường dẫn Kaggle
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORK  = Path('/kaggle/working')

VN_RAW_ROOT = KAGGLE_INPUT / VN_DATA_SLUG / 'dataset_aug'
STAGE2_PT   = KAGGLE_INPUT / MODEL_SLUG / STAGE2_PT_NAME
PROJECT_DIR = KAGGLE_WORK / 'yolov8_finetune_vn'

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Kiểm tra
print('=== Kiểm tra đường dẫn ===')
for p in [
    VN_RAW_ROOT / 'images',
    VN_RAW_ROOT / 'labels',
]:
    status = '✅' if p.exists() else '❌ MISSING'
    count  = len(list(p.glob('*'))) if p.exists() else 0
    print(f'{status}  {p}  ({count} files)')

s2_status = '✅' if STAGE2_PT.exists() else '❌ MISSING'
print(f'{s2_status}  {STAGE2_PT}')

if not STAGE2_PT.exists():
    print()
    print('⚠️  Không tìm thấy Stage 2 checkpoint. Các file .pt hiện có:')
    for pt in (KAGGLE_INPUT / MODEL_SLUG).rglob('*.pt'):
        print(f'   {pt}')

print(f'\n📁 Output dir: {PROJECT_DIR}')

## 3. Chia train / val / test từ dataset VN

In [ ]:
# Dataset VN chưa có sẵn split → tự động chia và copy sang /kaggle/working/

SPLIT_ROOT = PROJECT_DIR / 'dataset_split'

def build_split(force=False):
    if (SPLIT_ROOT / 'images' / 'train').exists() and not force:
        n_train = len(list((SPLIT_ROOT / 'images' / 'train').glob('*.*')))
        n_val   = len(list((SPLIT_ROOT / 'images' / 'val').glob('*.*')))
        n_test  = len(list((SPLIT_ROOT / 'images' / 'test').glob('*.*')))
        print(f'✅ Dataset đã chia: train={n_train} / val={n_val} / test={n_test}')
        print('   Đặt force=True để chia lại.')
        return

    img_dir = VN_RAW_ROOT / 'images'
    lbl_dir = VN_RAW_ROOT / 'labels'

    all_imgs = sorted(
        list(img_dir.rglob('*.jpg')) +
        list(img_dir.rglob('*.png')) +
        list(img_dir.rglob('*.jpeg'))
    )

    # Chỉ lấy ảnh có label tương ứng
    paired = []
    for img_path in all_imgs:
        lbl_path = lbl_dir / (img_path.stem + '.txt')
        if not lbl_path.exists():
            # Tìm đệ quy trong labels/
            candidates = list(lbl_dir.rglob(img_path.stem + '.txt'))
            lbl_path = candidates[0] if candidates else None
        if lbl_path and lbl_path.exists():
            paired.append((img_path, lbl_path))

    print(f'Tổng ảnh tìm thấy  : {len(all_imgs)}')
    print(f'Có label hợp lệ    : {len(paired)}')

    random.seed(SEED)
    random.shuffle(paired)

    n_train = int(len(paired) * TRAIN_RATIO)
    n_val   = int(len(paired) * VAL_RATIO)

    splits = {
        'train': paired[:n_train],
        'val'  : paired[n_train:n_train + n_val],
        'test' : paired[n_train + n_val:],
    }

    for split_name, items in splits.items():
        img_out = SPLIT_ROOT / 'images' / split_name
        lbl_out = SPLIT_ROOT / 'labels' / split_name
        img_out.mkdir(parents=True, exist_ok=True)
        lbl_out.mkdir(parents=True, exist_ok=True)
        for img_path, lbl_path in items:
            shutil.copy2(img_path, img_out / img_path.name)
            shutil.copy2(lbl_path, lbl_out / lbl_path.name)
        print(f'  [{split_name:>5}] {len(items):>5} ảnh  →  {img_out}')

    print('\n✅ Chia dataset hoàn tất.')


build_split(force=False)

## 4. Phân tích nhanh dataset

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from collections import Counter
from PIL import Image

# ⚠️ Giữ đúng thứ tự class từ Stage 2
CLASSES = ['car', 'truck', 'bus', 'motor']
COLORS  = ['#4C9BE8', '#E87B4C', '#4CE87B', '#E84C9B']

def count_labels(label_dir: Path):
    counter = Counter()
    n_files = 0
    for txt in label_dir.glob('*.txt'):
        n_files += 1
        for line in txt.read_text().strip().splitlines():
            if line.strip():
                cls = int(line.split()[0])
                counter[cls] += 1
    return counter, n_files

print('── Đang đếm nhãn... ──')
train_cnt, n_train = count_labels(SPLIT_ROOT / 'labels' / 'train')
val_cnt,   n_val   = count_labels(SPLIT_ROOT / 'labels' / 'val')
test_cnt,  n_test  = count_labels(SPLIT_ROOT / 'labels' / 'test')

for split_name, cnt, n in [
    ('TRAIN', train_cnt, n_train),
    ('VAL',   val_cnt,   n_val),
    ('TEST',  test_cnt,  n_test),
]:
    total = sum(cnt.values())
    print(f'\nTập {split_name}: {n:,} ảnh  |  {total:,} instances')
    for i, cls in enumerate(CLASSES):
        count = cnt[i]
        pct   = count / total * 100 if total else 0
        warn  = '  ⚠️ < 100' if (split_name != 'TRAIN' and count < 100) else \
                '  ⚠️ < 500' if (split_name == 'TRAIN' and count < 500) else ''
        print(f'  {cls:>6}: {count:>7,} ({pct:5.1f}%){warn}')

# ── Bar chart ──
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Class Distribution — VN Dataset', fontsize=13, fontweight='bold')

for ax, (split_name, cnt) in zip(axes, [
    ('Train', train_cnt), ('Val', val_cnt), ('Test', test_cnt)
]):
    vals = [cnt[i] for i in range(len(CLASSES))]
    bars = ax.bar(CLASSES, vals, color=COLORS, edgecolor='white', linewidth=0.5)
    ax.set_title(split_name, fontsize=11)
    ax.set_ylabel('Instances')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.01,
                f'{v:,}', ha='center', va='bottom', fontsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig(PROJECT_DIR / 'class_distribution_vn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Lưu class_distribution_vn.png')

## 5. Tạo data.yaml

In [ ]:
import yaml

data_yaml_path = PROJECT_DIR / 'vn_traffic.yaml'

data_yaml = {
    'path'  : str(SPLIT_ROOT),
    'train' : 'images/train',
    'val'   : 'images/val',
    'test'  : 'images/test',
    'nc'    : len(CLASSES),
    # ⚠️ Thứ tự class PHẢI khớp Stage 2: car=0, truck=1, bus=2, motor=3
    'names' : CLASSES,
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False, allow_unicode=True)

print(f'✅ Tạo {data_yaml_path}')
print()
print('Nội dung data.yaml:')
print('─' * 40)
print(open(data_yaml_path).read())

## 6. Visual check — xem ảnh mẫu với bbox

In [ ]:
train_imgs = list((SPLIT_ROOT / 'images' / 'train').glob('*.jpg')) + \
             list((SPLIT_ROOT / 'images' / 'train').glob('*.png'))
test_batch = random.sample(train_imgs, min(8, len(train_imgs)))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Sample Annotations — VN Train Set', fontsize=12, fontweight='bold')

for ax, img_path in zip(axes.flatten(), test_batch):
    img = Image.open(img_path).convert('RGB')
    W, H = img.size
    ax.imshow(img)

    lbl_path = SPLIT_ROOT / 'labels' / 'train' / (img_path.stem + '.txt')
    cls_counts = Counter()
    if lbl_path.exists():
        for line in open(lbl_path):
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cid = int(parts[0])
            xc, yc, w, h = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
            x1 = (xc - w / 2) * W
            y1 = (yc - h / 2) * H
            color = COLORS[cid % len(COLORS)]
            ax.add_patch(patches.Rectangle(
                (x1, y1), w * W, h * H,
                linewidth=1.5, edgecolor=color, facecolor='none'
            ))
            ax.text(x1, y1 - 3, CLASSES[cid] if cid < len(CLASSES) else str(cid),
                    color=color, fontsize=7, fontweight='bold')
            cls_counts[cid] += 1

    summary = ' '.join(f'{CLASSES[k]}:{v}' for k, v in sorted(cls_counts.items()))
    ax.set_title(summary if summary else 'no label', fontsize=8)
    ax.axis('off')

plt.tight_layout()
plt.savefig(PROJECT_DIR / 'sample_annotations_vn.png', dpi=120, bbox_inches='tight')
plt.show()
print('\n⚠️  Kiểm tra: bbox có bao đúng phương tiện không? class có đúng không?')

## 7. Load model Stage 2 & định nghĩa Gradual Unfreeze

In [ ]:
from ultralytics import YOLO
import cv2

# ── Load Stage 2 checkpoint ───────────────────────────────────────────────────
if not STAGE2_PT.exists():
    # Tìm tự động
    candidates = list((KAGGLE_INPUT / MODEL_SLUG).rglob('*.pt'))
    if candidates:
        STAGE2_PT = candidates[0]
        print(f'ℹ️  Tự động tìm: {STAGE2_PT}')
    else:
        raise FileNotFoundError(
            f'Không tìm thấy Stage 2 .pt trong {KAGGLE_INPUT / MODEL_SLUG}'
        )

print(f'✅ Stage 2 checkpoint : {STAGE2_PT}')
print(f'   Kích thước         : {STAGE2_PT.stat().st_size / 1e6:.1f} MB')


# ── Gradual Unfreeze Callback ─────────────────────────────────────────────────
# Stage 2 dùng freeze=10 toàn bộ. Stage 3 mở dần theo epoch:
#   Epoch  0–14 : freeze=10  (chỉ train head — adapt về VN distribution)
#   Epoch 15–34 : freeze=7   (mở neck + C2f cuối)
#   Epoch 35+   : freeze=3   (mở gần toàn bộ, lr vẫn thấp)

class GradualUnfreezeCallback:
    SCHEDULE = {15: 7, 35: 3}

    def __init__(self):
        self._current = 10
        self.log = []

    def on_train_epoch_start(self, trainer):
        ep = trainer.epoch
        for target_ep, new_freeze in sorted(self.SCHEDULE.items()):
            if ep == target_ep and self._current != new_freeze:
                self._current = new_freeze
                for i, layer in enumerate(trainer.model.model.children()):
                    for param in layer.parameters():
                        param.requires_grad = (i >= new_freeze)
                msg = f'🔓 Epoch {ep:>3}: unfreeze → freeze={new_freeze} layers'
                self.log.append(msg)
                print(msg)

    def on_train_end(self, trainer):
        if self.log:
            print('\n── Gradual unfreeze log ──')
            for m in self.log:
                print(f'  {m}')


print()
print('✅ GradualUnfreezeCallback sẵn sàng.')
print('   Schedule: epoch 15 → freeze=7 | epoch 35 → freeze=3')

## 8. Cấu hình & Training Stage 3

In [ ]:
# ─── STAGE 3 CONFIG ───────────────────────────────────────────────────────────
STAGE3_CONFIG = {
    # Data
    'data'        : str(data_yaml_path),
    'imgsz'       : 640,
    'batch'       : 16,          # Giảm xuống 8 nếu OOM
    'device'      : 0,

    # Schedule
    'epochs'      : 60,
    'patience'    : 15,          # Early stopping
    'freeze'      : 10,          # Bắt đầu freeze — callback mở dần

    # Optimizer (thấp hơn Stage 2 để không phá features đã học)
    'optimizer'   : 'AdamW',
    'lr0'         : 0.0005,      # Base lr cho head (5× thấp hơn Stage 2)
    'lrf'         : 0.01,        # Cosine decay: lr_end = lr0 × lrf
    'momentum'    : 0.937,
    'weight_decay': 0.001,       # Regularization — dataset nhỏ
    'warmup_epochs'   : 3,
    'warmup_momentum' : 0.8,

    # Augmentation — domain adaptation cho VN
    'mosaic'      : 1.0,         # Tạo cảnh đông đúc xe máy VN
    'mixup'       : 0.05,
    'copy_paste'  : 0.4,         # Bù truck/bus ít instance
    'close_mosaic': 10,          # Tắt mosaic ở 10 epoch cuối
    'hsv_h'       : 0.020,       # Hue jitter — ánh sáng đa dạng VN
    'hsv_s'       : 0.800,
    'hsv_v'       : 0.500,
    'degrees'     : 2.0,         # Camera cố định → ít cần rotate
    'translate'   : 0.10,
    'scale'       : 0.70,
    'shear'       : 1.5,
    'perspective' : 0.0003,
    'flipud'      : 0.0,         # Xe không lộn ngược
    'fliplr'      : 0.5,

    # Loss
    'box'         : 7.5,
    'cls'         : 0.5,
    'dfl'         : 1.5,
    'fl_gamma'    : 1.5,         # Focal loss — focus hard examples (xe máy nhỏ)

    # Output
    'project'     : str(PROJECT_DIR),
    'name'        : 'stage3_vn',
    'exist_ok'    : True,
    'save'        : True,
    'save_period' : 10,
    'val'         : True,
    'plots'       : True,
    'verbose'     : True,
}
# ──────────────────────────────────────────────────────────────────────────────

print('=== Stage 3 Config ===')
for k, v in STAGE3_CONFIG.items():
    print(f'  {k:<18}: {v}')

In [ ]:
# ─── TRAINING ─────────────────────────────────────────────────────────────────
torch.cuda.empty_cache()

model = YOLO(str(STAGE2_PT))

# Đăng ký callback
unfreeze_cb = GradualUnfreezeCallback()
model.add_callback('on_train_epoch_start', unfreeze_cb.on_train_epoch_start)
model.add_callback('on_train_end',          unfreeze_cb.on_train_end)

print('🚀 Bắt đầu Stage 3 — VN Finetune...')
print(f'   Base model : {STAGE2_PT}')
print(f'   Dataset    : {data_yaml_path}')
print(f'   Output     : {PROJECT_DIR}/stage3_vn/')
print()

train_results = model.train(**STAGE3_CONFIG)

best_weights = PROJECT_DIR / 'stage3_vn' / 'weights' / 'best.pt'
last_weights = PROJECT_DIR / 'stage3_vn' / 'weights' / 'last.pt'

print(f'\n✅ Training hoàn tất!')
print(f'   best.pt : {best_weights}  (exists={best_weights.exists()})')
print(f'   last.pt : {last_weights}  (exists={last_weights.exists()})')

## 9. Đánh giá kết quả trên tập Val

In [ ]:
# Đường dẫn best weights
best_weights = PROJECT_DIR / 'stage3_vn' / 'weights' / 'best.pt'
last_weights = PROJECT_DIR / 'stage3_vn' / 'weights' / 'last.pt'

print(f'Best weights: {best_weights}')
print(f'Exists      : {best_weights.exists()}')

# Load best model và validate
model_best = YOLO(str(best_weights))

val_results = model_best.val(
    data     = str(data_yaml_path),
    imgsz    = 640,
    batch    = 16,
    device   = 0,
    split    = 'val',
    save_json= False,
    plots    = True,
    verbose  = True,
)

print('\n' + '=' * 50)
print('  KẾT QUẢ ĐÁNH GIÁ STAGE 3 — VN Val')
print('=' * 50)
print(f'  mAP@0.5       : {val_results.box.map50:.4f}  (mục tiêu: ≥ 0.85)')
print(f'  mAP@0.5:0.95  : {val_results.box.map:.4f}')
print(f'  Precision     : {val_results.box.mp:.4f}')
print(f'  Recall        : {val_results.box.mr:.4f}')
print()

# Per-class breakdown
print('  Per-class mAP@0.5:')
for i, (cls_name, ap) in enumerate(zip(CLASSES, val_results.box.ap50)):
    bar  = '█' * int(ap * 30)
    warn = '  ⚠️ thấp' if ap < 0.65 else ''
    print(f'    {cls_name:>6}  {bar:<30}  {ap:.4f}{warn}')

## 10. Vẽ training curves

In [ ]:
import pandas as pd

results_csv = PROJECT_DIR / 'stage3_vn' / 'results.csv'

if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Stage 3 — Training Curves (VN Data)', fontsize=14, fontweight='bold')

    METRICS = [
        ('train/box_loss',       'Train Box Loss',   '#E84C4C'),
        ('train/cls_loss',       'Train Cls Loss',   '#E8944C'),
        ('train/dfl_loss',       'Train DFL Loss',   '#E8D44C'),
        ('metrics/mAP50(B)',     'mAP@0.5',          '#4CE87B'),
        ('metrics/precision(B)', 'Precision',         '#4C9BE8'),
        ('metrics/recall(B)',    'Recall',             '#9B4CE8'),
    ]

    for ax, (col, title, color) in zip(axes.flatten(), METRICS):
        if col in df.columns:
            ax.plot(df['epoch'], df[col], color=color, linewidth=2)
            is_loss  = 'loss' in col
            best_val = df[col].min() if is_loss else df[col].max()
            best_ep  = df.loc[df[col] == best_val, 'epoch'].values[0]
            ax.axvline(best_ep, color=color, linestyle='--', alpha=0.5)
            ax.set_title(f'{title}\nbest={best_val:.4f} @ ep{best_ep:.0f}', fontsize=10)
            ax.set_xlabel('Epoch')
            ax.grid(True, alpha=0.3)
            ax.spines[['top', 'right']].set_visible(False)

            # Vẽ unfreeze points
            for ep_mark, label in [(15, 'unfreeze\n→7'), (35, 'unfreeze\n→3')]:
                if ep_mark <= df['epoch'].max():
                    ax.axvline(ep_mark, color='gray', linestyle=':', alpha=0.6, linewidth=1)
                    ax.text(ep_mark + 0.3, ax.get_ylim()[0], label,
                            fontsize=7, color='gray', va='bottom')
        else:
            ax.text(0.5, 0.5, f'{col}\nnot found',
                    ha='center', va='center', transform=ax.transAxes)

    plt.tight_layout()
    save_path = PROJECT_DIR / 'training_curves_stage3.png'
    plt.savefig(str(save_path), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ Training curves lưu tại: {save_path}')
else:
    print('⚠️  results.csv chưa tồn tại — hãy chạy cell train trước')

## 11. Inference thử trên ảnh Val — kiểm tra visual

In [ ]:
val_imgs    = list((SPLIT_ROOT / 'images' / 'val').glob('*.jpg')) + \
              list((SPLIT_ROOT / 'images' / 'val').glob('*.png'))
test_batch  = random.sample(val_imgs, min(6, len(val_imgs)))

preds = model_best.predict(
    source  = test_batch,
    imgsz   = 640,
    conf    = 0.30,          # Thấp hơn default vì motor nhỏ
    iou     = 0.45,
    device  = 0,
    verbose = False,
)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Stage 3 — Inference Preview (VN Val Set, conf=0.30)', fontsize=13, fontweight='bold')

for ax, r in zip(axes.flatten(), preds):
    img = r.plot(line_width=2, font_size=10)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    cls_counts = Counter(int(c) for c in r.boxes.cls.cpu()) if r.boxes else Counter()
    summary    = '  '.join(f'{CLASSES[k]}:{v}' for k, v in sorted(cls_counts.items()))
    ax.set_title(summary if summary else 'no detection', fontsize=9)
    ax.axis('off')

plt.tight_layout()
save_path = PROJECT_DIR / 'inference_preview_stage3.png'
plt.savefig(str(save_path), dpi=120, bbox_inches='tight')
plt.show()
print(f'✅ Preview lưu tại: {save_path}')

## 12. Đánh giá trên Test Set (hold-out — chỉ mở 1 lần)

In [ ]:
# ⚠️ Test set chỉ được dùng 1 lần khi báo cáo kết quả cuối cùng
# Bỏ comment và chạy cell này khi đã chắc chắn không thay đổi model nữa

# test_yaml_path = PROJECT_DIR / 'vn_test_eval.yaml'
# import yaml
# test_yaml = dict(data_yaml)
# test_yaml['val'] = 'images/test'
# with open(test_yaml_path, 'w') as f:
#     yaml.dump(test_yaml, f, default_flow_style=False)
#
# test_results = model_best.val(
#     data    = str(test_yaml_path),
#     split   = 'val',
#     imgsz   = 640,
#     batch   = 16,
#     device  = 0,
#     conf    = 0.001,
#     verbose = True,
#     save_json = True,
# )
# print(f'TEST mAP@0.5     : {test_results.box.map50:.4f}')
# print(f'TEST mAP@0.5:0.95: {test_results.box.map:.4f}')
# for cls_name, ap in zip(CLASSES, test_results.box.ap50):
#     print(f'  {cls_name:>6}: {ap:.4f}')

print('ℹ️  Cell này bị comment — bỏ comment khi muốn đánh giá TEST SET lần cuối.')

## 13. Tóm tắt & chuẩn bị kết quả

In [ ]:
import json

results_csv  = PROJECT_DIR / 'stage3_vn' / 'results.csv'
epochs_done  = 0
if results_csv.exists():
    df_res      = pd.read_csv(results_csv)
    df_res.columns = df_res.columns.str.strip()
    epochs_done = int(df_res['epoch'].max())

# Lưu summary JSON
summary = {
    'stage'          : 3,
    'base_model'     : str(STAGE2_PT),
    'dataset'        : 'VN Traffic — dataset_aug',
    'classes'        : CLASSES,
    'epochs_trained' : epochs_done,
    'best_weights'   : str(best_weights),
    'metrics': {
        'mAP50'     : round(float(val_results.box.map50), 4),
        'mAP50_95'  : round(float(val_results.box.map),   4),
        'precision' : round(float(val_results.box.mp),    4),
        'recall'    : round(float(val_results.box.mr),    4),
        'per_class' : {
            name: round(float(ap), 4)
            for name, ap in zip(CLASSES, val_results.box.ap50)
        },
    },
}

summary_path = PROJECT_DIR / 'stage3_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# Copy best.pt về root /kaggle/working/ để dễ download
dst_best = KAGGLE_WORK / 'stage3_best.pt'
shutil.copy2(best_weights, dst_best)

print('╔══════════════════════════════════════════════════╗')
print('║           STAGE 3 — HOÀN THÀNH ✅               ║')
print('╠══════════════════════════════════════════════════╣')
print(f'║  mAP@0.5      : {val_results.box.map50:.4f}  (target ≥ 0.85)          ║')
print(f'║  mAP@0.5:0.95 : {val_results.box.map:.4f}                        ║')
print(f'║  Precision    : {val_results.box.mp:.4f}                        ║')
print(f'║  Recall       : {val_results.box.mr:.4f}                        ║')
print('╠══════════════════════════════════════════════════╣')
print('  Per-class mAP@0.5:')
for cls_name, ap in zip(CLASSES, val_results.box.ap50):
    bar  = '█' * int(ap * 20)
    print(f'    {cls_name:>6}  {bar:<20}  {ap:.4f}')
print('╠══════════════════════════════════════════════════╣')
print(f'║  Best weights : stage3_best.pt (root)            ║')
print(f'║  Summary JSON : stage3_summary.json              ║')
print('╠══════════════════════════════════════════════════╣')
print('║  BƯỚC TIẾP THEO:                                 ║')
print('║  Dùng best.pt → tích hợp ByteTrack tracking      ║')
print('║  Chạy calibration threshold per-class            ║')
print('║  Export ONNX → TensorRT FP16 để deploy           ║')
print('╚══════════════════════════════════════════════════╝')
print()
print('📂 Files trong /kaggle/working/:')
for f in sorted(KAGGLE_WORK.glob('*.pt')) + \
         sorted(KAGGLE_WORK.glob('*.json')):
    size = f.stat().st_size / 1e6
    print(f'  {f.name}  ({size:.1f} MB)')